# Semana 06: Containerização de Aplicações com Docker e Multi-Stage Builds

## Módulo de Containerização — Fábrica Virtual Smart N1

Este notebook apresenta os conceitos fundamentais de **containerização com Docker**, o isolamento no nível de Kernel (Namespaces e Cgroups), a criação de **Dockerfiles otimizados** e a técnica de **Multi-Stage Builds** para produção.

### Objetivos de aprendizagem
- Compreender as diferenças arquiteturais entre Máquinas Virtuais (VMs) e Containers Docker.
- Dominar os comandos e a sintaxe fundamental do `Dockerfile` (`FROM`, `WORKDIR`, `COPY`, `RUN`, `CMD`, `ENTRYPOINT`).
- Entender o funcionamento do sistema de camadas (*Layer Caching*) do Docker.
- Aplicar a técnica de **Multi-Stage Builds** para reduzir drasticamente o tamanho das imagens em produção.
- Desenvolver um gerador e otimizador de Dockerfiles em Python.

---


## 1. Fundamentação Teórica

### 1.1 Máquinas Virtuais vs Containers Docker

```text
  +-------------------------------+     +-------------------------------+
  |    MÁQUINAS VIRTUAIS (VMs)    |     |       CONTAINERS DOCKER       |
  | +-----------+   +-----------+ |     | +-----------+   +-----------+ |
  | | App A     |   | App B     | |     | | App A     |   | App B     | |
  | | Bins/Libs |   | Bins/Libs | |     | | Bins/Libs |   | Bins/Libs | |
  | | Guest OS  |   | Guest OS  | |     | +-----------+   +-----------+ |
  | +-----------+   +-----------+ |     | |      Docker Engine          | |
  | |        Hypervisor         | |     | +---------------------------+ |
  | +---------------------------+ |     | |  Kernel do SO Hospedeiro  | |
  | |  Kernel do SO Hospedeiro  | |     | +---------------------------+ |
  +-------------------------------+     +-------------------------------+
```

- **Máquina Virtual:** Virtualiza o **hardware completo**. Cada VM inclui uma cópia completa de um Sistema Operacional Convidado (*Guest OS*), consumindo gigabytes de RAM e disco.
- **Container Docker:** Virtualiza apenas o **Sistema Operacional**. Todos os containers compartilham o mesmo Kernel do hospedeiro (*Host OS*), garantindo inicialização em milissegundos e tamanho reduzido.

---

### 1.2 Otimização com Multi-Stage Builds

O **Multi-Stage Build** permite utilizar múltiplas instruções `FROM` em um único `Dockerfile`. Cada estágio pode usar uma imagem de base diferente e copiar apenas os artefatos compilados necessários para a imagem final:

```text
  [Estágio 1: Builder (python:3.11-slim)]       [Estágio 2: Runner (python:3.11-alpine)]
  +-------------------------------------+       +--------------------------------------------+
  | Instala gcc, pip, pytest, headers   |       | Contém apenas o interpretador Python leve   |
  | Compila dependências pesadas        |       | sem ferramentas de compilação ou shell     |
  | Gerador de artefatos / wheels       |       +--------------------------------------------+
  +-------------------------------------+                             ^
                     |                                                |
                     +--- COPY --from=builder /app/wheels ----------> +
```

---

### 1.3 Exemplo de Dockerfile Multi-Stage para Aplicação Fabril

```dockerfile
# Estágio 1: Construtor (Builder)
FROM python:3.11-slim AS builder

WORKDIR /app
RUN apt-get update && apt-get install -y gcc libpq-dev

COPY requirements.txt .
RUN pip wheel --no-cache-dir --no-deps --wheel-dir /app/wheels -r requirements.txt

# Estágio 2: Execução Final (Runner Leve)
FROM python:3.11-alpine AS runner

WORKDIR /app
RUN addgroup -S appgroup && adduser -S appuser -G appgroup

COPY --from=builder /app/wheels /wheels
RUN pip install --no-cache /wheels/*

COPY . /app
USER appuser

EXPOSE 5000
CMD ["python", "app.py"]
```

---


## 2. Prática — Gerador e Otimizador de Dockerfiles em Python

Nesta atividade prática, utilizaremos um script Python para comparar o tamanho estimado de uma imagem convencional monolítica contra uma imagem criada via Multi-Stage Build.

In [ ]:
def analisar_estatisticas_imagem_docker(nome_app, libs_compilacao, codigo_fonte_mb):
    # Estimativas em MB
    tamanho_python_full = 920.0
    tamanho_python_alpine = 50.0
    tamanho_compiladores = 350.0
    tamanho_wheels = 85.0
    
    # 1. Imagem Monolítica Sem Multi-Stage
    tamanho_monolitico = tamanho_python_full + tamanho_compiladores + tamanho_wheels + codigo_fonte_mb
    
    # 2. Imagem Otimizada com Multi-Stage (Estágio final Alpine)
    tamanho_multi_stage = tamanho_python_alpine + tamanho_wheels + codigo_fonte_mb
    
    reducao_pct = round(((tamanho_monolitico - tamanho_multi_stage) / tamanho_monolitico) * 100.0, 1)
    
    return {
        "aplicacao": nome_app,
        "tamanho_monolitico_mb": round(tamanho_monolitico, 1),
        "tamanho_multi_stage_mb": round(tamanho_multi_stage, 1),
        "economia_espaco_mb": round(tamanho_monolitico - tamanho_multi_stage, 1),
        "reducao_percentual": f"{reducao_pct}%"
    }

estatisticas = analisar_estatisticas_imagem_docker("SmartN1_Telemetria_API", ["gcc", "g++", "make"], 15.0)

print("=== COMPARAÇÃO DE IMPACTO DE MULTI-STAGE BUILDS ===\n")
for k, v in estatisticas.items():
    print(f"- {k}: {v}")


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1
Por que a ordem das instruções em um `Dockerfile` afeta o reaproveitamento de cache de camadas (*layer caching*)? Dê um exemplo de por que copiar os arquivos de código-fonte antes de instalar os pacotes (pip/npm) é uma má prática.

### Questão 2
Explique a diferença entre `CMD` e `ENTRYPOINT` no Dockerfile. Como eles interagem quando executamos um container passando argumentos adicionais no comando `docker run`?

### Questão 3
Qual a vantagem de segurança em definir a instrução `USER appuser` no Dockerfile em vez de permitir que a aplicação execute com o usuário `root` padrão dentro do container?
